# Lucraftmeds team crawler (Kaggle + direct Google Drive API)

Each worker crawls its own dataset shard. JSONL checkpoints are uploaded directly from RAM to the shared Drive folder after each batch; crawl output is never written to local disk. Each checkpoint contains both success and error records (`success` distinguishes them).

Before running: enable Internet in Kaggle and attach the `GOOGLE_DRIVE_TOKEN` secret containing your OAuth `token.json`. The OAuth account must have write access to the folder. Use a different `WORKER_ID` per active teammate, with identical dataset/sharding settings. Previous-day URL history is shared across all shards; it does not depend on the current `WORKER_ID`.

A terminated runtime may lose the current, not-yet-uploaded batch (at most `BATCH_SIZE` URLs); those URLs are crawled again on resume. Previously uploaded batches are preserved. On a fresh runtime, only yesterday's merged URL history is used to skip old work, so today's batches may be recrawled until aggregated. OAuth External/Testing refresh tokens normally expire after seven days; renew the secret if needed.

## 1. Install dependencies
Run once per fresh Kaggle runtime. Internet must be enabled.

In [ ]:
%pip install -q -U crawl4ai datasets tqdm pandas google-api-python-client google-auth
!crawl4ai-setup


## 2. Connect to Google Drive and configure this worker
Attach `GOOGLE_DRIVE_TOKEN` in Kaggle Add-ons → Secrets. It must contain the complete OAuth token JSON, not the desktop client credentials. No Colab mount is needed. Never print the secret.

For two teammates, set `NUM_WORKERS = 2`, with `WORKER_ID = 0` or `1`.

In [1]:
import json
from kaggle_secrets import UserSecretsClient
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build

credentials = Credentials.from_authorized_user_info(
    json.loads(UserSecretsClient().get_secret("GOOGLE_DRIVE_TOKEN"))
)
if not credentials.valid:
    if not credentials.refresh_token:
        raise RuntimeError("Secret is missing a refresh token. Authorize again locally.")
    credentials.refresh(Request())

drive_service = build("drive", "v3", credentials=credentials, cache_discovery=False)
user = drive_service.about().get(fields="user(emailAddress)").execute()
print("OAuth account:", user["user"].get("emailAddress"))


ModuleNotFoundError: No module named 'kaggle_secrets'

In [ ]:
# All teammates use the same values except WORKER_ID.
NUM_WORKERS = 25
WORKER_ID = 1                 # Use 0 or 1; each active teammate needs a different ID.
DRIVE_FOLDER_ID = "13B_2bZOTBbmi-3UIy1oQwW1fQU610Jrh"

HISTORY_PARENT_NAME = "results"  # Folder containing DD-MM-YYYY/success.jsonl and errors.jsonl.

MAX_ITEMS = None              # Example: 100 for a quick test.
RETRY_FAILED = False          # True retries yesterday's errors; False skips their URLs too.
BATCH_SIZE = 50               # URLs submitted to each arun_many call.
MAX_CONCURRENCY = 5           # Start at 5 on Kaggle; raise only after measuring RAM/errors.
MEMORY_THRESHOLD_PERCENT = 80.0

# Only these domains use slower anti-bot settings. Keep the normal path fast.
PROTECTED_DOMAINS = {
    'special.vietnamplus.vn',
    'cnkang.com',
    '120ask.com',
    'familydoctor.com.cn',
    'ask.39.net',
    'vov2.vov.vn',
    'a-hospital.com',
    'suckhoecongdongonline.vn',
    'zhongyibaodian.net',
    'suckhoecongdongonline.vn',
    'zydcd.com',
    'thanhnien.vn',
    'wujue.com',
    'youlai.cn',
    'vov.vn',
    'zysjonline.com',
    'pmc-ecm-healthblog.beta.pharmacity.io',
    'nhathuoclongchau.com.vn',
    'giadinhonline.vn',
    'ask.familydoctor.com.cn',
}

SLOW_DOMAINS = {
    'bachmai.gov.vn',
    'wujue.com',
    'test.pmphai.com',
    'qihuangzhishu.com',
    'woman.39.net',
    'tnb.39.net',
    'wei.39.net',
    'shen.39.net',
    'baidu.com',
    'sj.39.net',
    'v.familydoctor.com.cn',
    'yanglao.familydoctor.com.cn',
    'ypk.familydoctor.com.cn',
    'baothanhhoa.vn',
    
}

if not 0 <= WORKER_ID < NUM_WORKERS:
    raise ValueError('WORKER_ID must be in range [0, NUM_WORKERS).')

folder = drive_service.files().get(
    fileId=DRIVE_FOLDER_ID,
    supportsAllDrives=True,
    fields="id,name,mimeType,capabilities(canAddChildren)",
).execute()
if folder["mimeType"] != "application/vnd.google-apps.folder":
    raise ValueError("DRIVE_FOLDER_ID must refer to a folder, not a file.")
if not folder.get("capabilities", {}).get("canAddChildren", False):
    raise PermissionError("The OAuth account cannot add files to this folder.")

print(f"Worker: {WORKER_ID}/{NUM_WORKERS - 1}")
print(f"Direct Drive output: {folder['name']} ({DRIVE_FOLDER_ID})")


## 3. Load and shard the dataset
Each URL belongs to exactly one worker when everyone uses the same configuration.

In [2]:
from datasets import load_dataset

ds = load_dataset('AIGuruTinix/ViBioMIR', 'corpus')
urls = ds['train'].shard(
    num_shards=NUM_WORKERS,
    index=WORKER_ID,
    contiguous=True,
)
if MAX_ITEMS is not None:
    urls = urls.select(range(min(MAX_ITEMS, len(urls))))
print(f'This worker owns {len(urls):,} URLs.')


/Users/buithianhdao/.pyenv/versions/3.10.16/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'NUM_WORKERS' is not defined

## 4. Direct Drive checkpoints and yesterday's URL history
Skip history is read from `results/DD-MM-YYYY/success.jsonl`, where the date is yesterday in Vietnam (`Asia/Ho_Chi_Minh`). `HISTORY_PARENT_NAME` defaults to `results` to match the aggregation notebook; change it to `data` only if your dated folders actually live there.

The history combines all workers. URL matching uses the original `url` field (trimmed whitespace), not `doc_id`, final redirect URL, or shard membership. When `RETRY_FAILED = False`, `errors.jsonl` is included too; when True, only successful URLs are skipped. Missing parent/date folders mean no previous-day history and are reported. A date folder without `success.jsonl` is treated as incomplete and stops the notebook.

Today's batch upload format is unchanged. Only confirmed uploads add current-session IDs and URLs to the skip sets. This notebook no longer loads old worker checkpoint files for resume; after a fresh runtime restart, today's batches are not skipped until included in the previous-day aggregate.

History JSONL is downloaded into an automatically cleaned temporary file to avoid holding the full merged Markdown file in RAM. Only its URL set stays in memory. Batch output is still uploaded directly from RAM. Upload failures retain `PENDING_UPLOAD` for retry as before.

In [ ]:
import json
import io
import uuid
import tempfile
from datetime import datetime, timezone, timedelta
from zoneinfo import ZoneInfo
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaIoBaseUpload, MediaIoBaseDownload


def list_drive_files(folder_id):
    page_token = None
    while True:
        page = drive_service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields="nextPageToken,files(id,name,mimeType)",
            pageSize=1000,
            pageToken=page_token,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        ).execute(num_retries=3)
        yield from page.get("files", [])
        page_token = page.get("nextPageToken")
        if not page_token:
            break


def worker_checkpoint_files(worker_id):
    # Used only by the optional preview, not to construct the skip history.
    prefix = f"worker_{worker_id:02d}_batch_"
    for file in list_drive_files(DRIVE_FOLDER_ID):
        if file["name"].startswith(prefix) and file["name"].endswith(".jsonl"):
            yield file


def read_drive_jsonl(file):
    with tempfile.TemporaryFile(mode="w+b") as buffer:
        download = MediaIoBaseDownload(
            buffer,
            drive_service.files().get_media(fileId=file["id"], supportsAllDrives=True),
        )
        done = False
        while not done:
            _, done = download.next_chunk(num_retries=3)
        buffer.seek(0)
        for line_number, line in enumerate(buffer, 1):
            if line.strip():
                try:
                    yield json.loads(line)
                except (ValueError, UnicodeError) as exc:
                    # Fail closed: malformed checkpoints must not be silently ignored.
                    raise ValueError(
                        f"Invalid JSONL in {file['name']}, line {line_number}"
                    ) from exc


def find_child_folder(parent_id, name):
    matches = [file for file in list_drive_files(parent_id)
               if file["name"] == name
               and file["mimeType"] == "application/vnd.google-apps.folder"]
    if len(matches) > 1:
        raise ValueError(f"Multiple folders named {name!r}; resolve duplicate names first.")
    return matches[0] if matches else None


def load_completed_urls():
    today = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).date()

    parent = find_child_folder(DRIVE_FOLDER_ID, HISTORY_PARENT_NAME)
    if parent is None:
        print(f"Không có folder {HISTORY_PARENT_NAME}; chưa có lịch sử skip.")
        return set()

    # Tìm hôm nay trước.
    day_name = today.strftime("%d-%m-%Y")
    day_folder = find_child_folder(parent["id"], day_name)

    # Chỉ chuyển sang hôm qua nếu folder hôm nay không tồn tại.
    if day_folder is None:
        day_name = (today - timedelta(days=1)).strftime("%d-%m-%Y")
        day_folder = find_child_folder(parent["id"], day_name)

    if day_folder is None:
        print("Không có folder hôm nay hoặc hôm qua; chưa có lịch sử skip.")
        return set()

    history_path = f"{HISTORY_PARENT_NAME}/{day_name}"
    print("Đọc lịch sử:", history_path)

    files = list(list_drive_files(day_folder["id"]))
    names = ["success.jsonl"]
    if not RETRY_FAILED:
        names.append("errors.jsonl")

    completed = set()

    for name in names:
        matches = [file for file in files if file["name"] == name]

        if len(matches) > 1:
            raise ValueError(f"Trùng file {history_path}/{name}")

        if not matches:
            if name == "success.jsonl":
                raise FileNotFoundError(
                    f"Thiếu {history_path}/{name}. Hãy chạy notebook gộp."
                )
            print(f"Không có {history_path}/{name}; bỏ qua file này.")
            continue

        for row in read_drive_jsonl(matches[0]):
            url = row.get("url")
            if not isinstance(url, str) or not url.strip():
                raise ValueError(f"Bản ghi thiếu URL trong {history_path}/{name}")
            completed.add(url.strip())

    return completed


# Preserve an unsent batch if this helper cell is rerun in the same runtime.
PENDING_UPLOAD = globals().get("PENDING_UPLOAD")


def retry_pending_upload():
    global PENDING_UPLOAD
    pending = PENDING_UPLOAD
    if pending is None:
        return None
    with io.BytesIO(pending["content"]) as buffer:
        try:
            uploaded = drive_service.files().create(
                body={"id": pending["id"], "name": pending["name"],
                      "parents": [DRIVE_FOLDER_ID]},
                media_body=MediaIoBaseUpload(
                    buffer, mimetype="application/x-ndjson", resumable=True,
                ),
                supportsAllDrives=True,
                fields="id,name",
            ).execute(num_retries=3)
        except HttpError as exc:
            if exc.resp.status != 409:
                raise
            # The previous request may have succeeded before its response was lost.
            uploaded = drive_service.files().get(
                fileId=pending["id"], supportsAllDrives=True, fields="id,name,size,parents"
            ).execute(num_retries=3)
            if (uploaded.get("name") != pending["name"] or
                    int(uploaded.get("size", -1)) != len(pending["content"]) or
                    DRIVE_FOLDER_ID not in uploaded.get("parents", [])):
                raise RuntimeError("Existing file does not match the pending checkpoint.") from exc
    completed_ids.update(pending["doc_ids"])
    completed_urls.update(pending["urls"])
    PENDING_UPLOAD = None
    print(f"Saved Drive checkpoint: {uploaded['name']}")
    return uploaded


def upload_batch(records):
    global PENDING_UPLOAD
    if PENDING_UPLOAD is not None:
        raise RuntimeError("Retry PENDING_UPLOAD before starting another batch.")
    if not records:
        return None
    content = "".join(json.dumps(row, ensure_ascii=False, default=str) + "\n"
                      for row in records).encode("utf-8")
    file_id = drive_service.files().generateIds(count=1, space="drive").execute(
        num_retries=3
    )["ids"][0]
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
    PENDING_UPLOAD = {
        "id": file_id,
        "name": f"worker_{WORKER_ID:02d}_batch_{stamp}_{uuid.uuid4().hex}.jsonl",
        "content": content,
        "doc_ids": {str(row["doc_id"]) for row in records},
        "urls": {row["url"].strip() for row in records},
    }
    return retry_pending_upload()



completed_urls = load_completed_urls()
completed_ids = set()  # IDs uploaded during this runtime only.
print(f"Skipping {len(completed_urls):,} previous-day URLs across all workers.")


## 5. Crawl4AI configuration
One browser instance is reused for the complete worker shard. Normal pages use a fast configuration; only domains in `PROTECTED_DOMAINS` use the slower anti-bot configuration.

In [ ]:
def first_metadata_value(metadata: dict, *keys):
    for key in keys:
        value = metadata.get(key)

        if isinstance(value, str) and value.strip():
            return value.strip()

    return None


In [ ]:
from urllib.parse import urlparse, urlsplit

from crawl4ai import (
    AsyncWebCrawler,
    BrowserConfig,
    CacheMode,
    CrawlerRunConfig,
    MemoryAdaptiveDispatcher,
    RateLimiter,
)
from crawl4ai.content_filter_strategy import PruningContentFilterLXML
from crawl4ai.markdown_generation_strategy import DefaultMarkdownGenerator

browser_config = BrowserConfig(
    browser_type='chromium',
    headless=True,
    enable_stealth=True,
    user_agent_mode='random',
)

prune_filter = PruningContentFilterLXML(
    threshold=0.5,
    threshold_type='fixed',
    min_word_threshold=15,
)

markdown_generator = DefaultMarkdownGenerator(
    content_source='cleaned_html',
    content_filter=prune_filter,
    options={'ignore_links': True},
)

# Fast default for most URLs. Streaming lets us checkpoint each completed result.
fast_config = CrawlerRunConfig(
    stream=True,
    cache_mode=CacheMode.BYPASS,
    wait_until='domcontentloaded',
    page_timeout=45_000,
    delay_before_return_html=0.2,
    markdown_generator=markdown_generator,
)

# Expensive behavior is restricted to known protected domains.
protected_config = CrawlerRunConfig(
    url_matcher=lambda url: urlparse(url).netloc.lower().removeprefix('www.') in PROTECTED_DOMAINS,
    stream=True,
    cache_mode=CacheMode.BYPASS,
    magic=True,
    simulate_user=True,
    override_navigator=True,
    page_timeout=90_000,
    delay_before_return_html=3.0,
    max_retries=2,
    markdown_generator=markdown_generator,
)

slow_config = CrawlerRunConfig(
    url_matcher=lambda url: (
        urlparse(url).netloc
        .lower()
        .removeprefix("www.") in SLOW_DOMAINS
    ),

    stream=True,
    cache_mode=CacheMode.BYPASS,

    # Chỉ chờ server bắt đầu trả document,
    # không bắt buộc DOMContentLoaded phải hoàn tất.
    wait_until="commit",

    page_timeout=90_000,

    # Sau commit, đợi nội dung body thực sự xuất hiện.
    wait_for="""
        js:() => {
            return document.body
                && document.body.innerText
                && document.body.innerText.length > 500;
        }
    """,
    wait_for_timeout=30_000,
    delay_before_return_html=1.0,

    markdown_generator=markdown_generator,
)

crawl_configs = [
    protected_config,  # Anti-bot domains
    slow_config,       # Slow-loading domains
    fast_config,       # Default
]

def make_dispatcher():
    return MemoryAdaptiveDispatcher(
        memory_threshold_percent=MEMORY_THRESHOLD_PERCENT,
        max_session_permit=MAX_CONCURRENCY,
        rate_limiter=RateLimiter(
            base_delay=(0.2, 0.8),
            max_delay=20.0,
            max_retries=2,
        ),
    )

def markdown_text(markdown_result):
    if markdown_result is None:
        return ""

    if isinstance(markdown_result, str):
        return markdown_result

    raw = markdown_result.raw_markdown or ""
    fit = markdown_result.fit_markdown or ""

    # FIT chỉ chứa "word" lặp lại → dùng RAW.
    if fit.strip() and set(fit.split()) != {"word"}:
        return fit

    return raw

def url_identity(url: str) -> tuple[str, str, str]:
    parsed = urlsplit(url or '')
    host = parsed.netloc.lower().removeprefix('www.')
    path = parsed.path.rstrip('/') or '/'
    return host, path, parsed.query

def blocked_page_reason(title, markdown, status_code):
    title_text = (title or '').strip().lower()
    content = (markdown or '').strip().lower()
    if status_code in {403, 429, 503}:
        return f'http_{status_code}'
    blocked_titles = {'checking your browser...', 'just a moment...', 'access denied'}
    if title_text in blocked_titles:
        return 'anti_bot_challenge'
    markers = ('checking your browser', 'verify you are human', 'cf-chl-', 'captcha')
    if len(content) < 2_000 and any(marker in content for marker in markers):
        return 'anti_bot_challenge'
    return None

def crawl_result_to_record(result, requested_url: str) -> dict:
    status_code = getattr(result, 'status_code', None)
    redirected_url = getattr(result, 'redirected_url', None)
    final_url = redirected_url or getattr(result, 'url', None) or requested_url
    final_status_code = getattr(result, 'redirected_status_code', None) or status_code
    base = {
        'url': requested_url,
        'final_url': final_url,
        'domain': urlparse(final_url).netloc.lower(),
        'status_code': status_code,
        'final_status_code': final_status_code,
        'redirected': final_url != requested_url,
    }
    if not result.success:
        return {
            **base,
            'success': False,
            'error_type': 'crawl_error',
            'error': getattr(result, 'error_message', None) or 'Unknown crawl error',
        }

    metadata = getattr(result, 'metadata', None) or {}
    title = first_metadata_value(metadata, 'title', 'og:title', 'twitter:title')
    description = first_metadata_value(
        metadata, 'description', 'og:description', 'twitter:description'
    )
    keywords = first_metadata_value(metadata, 'keywords')
    markdown = markdown_text(getattr(result, 'markdown', None))
    blocked_reason = blocked_page_reason(title, markdown, final_status_code)

    if blocked_reason:
        return {
            **base,
            'title': title,
            'description': description,
            'keywords': keywords,
            'markdown': markdown,
            'success': False,
            'retryable': True,
            'error_type': blocked_reason,
            'error': 'Received an anti-bot or rate-limit page instead of article content.',
        }
    if not markdown.strip():
        return {
            **base,
            'title': title,
            'success': False,
            'retryable': True,
            'error_type': 'empty_content',
            'error': 'Crawl succeeded but produced no Markdown.',
        }
    return {
        **base,
        'title': title,
        'description': description,
        'keywords': keywords,
        'markdown': markdown,
        'success': True,
    }


## 6. Run this worker
The dataset shard remains unchanged. A URL is skipped if its original `url` is in yesterday's merged history, regardless of which worker previously crawled it. Current-session uploaded IDs/URLs are also skipped. Matching is exact apart from leading/trailing whitespace.

Results are collected for one bounded batch and uploaded directly to Drive in the existing mixed success/error format. No changes to the aggregation notebook are needed.

`RETRY_FAILED = False` includes yesterday's `errors.jsonl` in the skip set. Set True to retry failures; URLs also present in `success.jsonl` are still skipped. A new runtime only reads yesterday's history, not today's batch files.

In [ ]:
from collections import defaultdict, deque
from tqdm.auto import tqdm

def pending_lookup(batch):
    lookup = defaultdict(deque)
    for item in batch:
        lookup[url_identity(item['url'])].append(item)
    return lookup

def remaining_items(lookup):
    for queue in lookup.values():
        yield from queue

async def crawl_batch(crawler, dispatcher, batch, progress):
    records = []
    lookup = pending_lookup(batch)
    successes = errors = unmatched_results = 0

    try:
        result_stream = await crawler.arun_many(
            urls=[item['url'] for item in batch],
            config=crawl_configs,
            dispatcher=dispatcher,
        )

        async for crawl_result in result_stream:
            result_url = getattr(crawl_result, 'url', None) or ''
            queue = lookup.get(url_identity(result_url))
            if not queue:
                # Do not guess a doc_id when a returned URL cannot be mapped safely.
                unmatched_results += 1
                continue

            item = queue[0]
            doc_id = str(item['id'])
            record = crawl_result_to_record(crawl_result, item['url'])
            record['doc_id'] = doc_id
            queue.popleft()

            if record['success']:
                records.append(record)
                successes += 1
            else:
                records.append(record)
                errors += 1

            progress.update(1)

    except Exception as exc:
        batch_error = f'{type(exc).__name__}: {exc}'
        for item in remaining_items(lookup):
            doc_id = str(item['id'])
            record = {
                'doc_id': doc_id,
                'url': item['url'],
                'success': False,
                'retryable': True,
                'error_type': 'batch_error',
                'error': batch_error,
            }
            records.append(record)
            errors += 1
            progress.update(1)
        return records, successes, errors, unmatched_results

    # A missing/unmappable result is retried later instead of being linked to the wrong doc_id.
    for item in remaining_items(lookup):
        doc_id = str(item['id'])
        record = {
            'doc_id': doc_id,
            'url': item['url'],
            'success': False,
            'retryable': True,
            'error_type': 'unmapped_result',
            'error': 'arun_many returned no safely mappable result for this URL.',
        }
        records.append(record)
        errors += 1
        progress.update(1)

    return records, successes, errors, unmatched_results

import asyncio

# Retry a failed upload retained in this runtime before crawling anything else.
if PENDING_UPLOAD is not None:
    await asyncio.to_thread(retry_pending_upload)
    LAST_BATCH_RECORDS = None
elif globals().get("LAST_BATCH_RECORDS"):
    await asyncio.to_thread(upload_batch, LAST_BATCH_RECORDS)
    LAST_BATCH_RECORDS = None

new_successes = new_errors = skipped = unmatched_results = 0
batch = []
dispatcher = make_dispatcher()
progress = tqdm(total=len(urls), desc=f"Worker {WORKER_ID} crawling")

async def process_and_checkpoint(batch):
    # Retain records even if allocating an upload ID fails.
    global LAST_BATCH_RECORDS
    records, ok, failed, unmatched = await crawl_batch(
        crawler, dispatcher, batch, progress
    )
    LAST_BATCH_RECORDS = records
    await asyncio.to_thread(upload_batch, records)
    LAST_BATCH_RECORDS = None
    return ok, failed, unmatched

try:
    async with AsyncWebCrawler(config=browser_config) as crawler:
        for item in urls:
            doc_id = str(item["id"])
            if doc_id in completed_ids or item["url"].strip() in completed_urls:
                skipped += 1
                progress.update(1)
                continue
            batch.append({"id": doc_id, "url": item["url"]})
            if len(batch) < BATCH_SIZE:
                continue
            ok, failed, unmatched = await process_and_checkpoint(batch)
            new_successes += ok
            new_errors += failed
            unmatched_results += unmatched
            progress.set_postfix(ok=new_successes, errors=new_errors)
            batch = []
        if batch:
            ok, failed, unmatched = await process_and_checkpoint(batch)
            new_successes += ok
            new_errors += failed
            unmatched_results += unmatched
except Exception:
    if PENDING_UPLOAD is not None:
        print("Upload failed. Batch retained in PENDING_UPLOAD; rerun this cell to retry.")
    elif globals().get("LAST_BATCH_RECORDS"):
        print("Batch retained in LAST_BATCH_RECORDS. Run upload_batch(LAST_BATCH_RECORDS) to save it.")
    raise
finally:
    progress.close()

print({"new_successes": new_successes, "new_errors": new_errors,
       "skipped_existing": skipped, "unmatched_results": unmatched_results})


## 7. Inspect this worker's results (optional)
Preview a bounded sample directly from Drive without loading the entire corpus into a DataFrame. A batch includes successes and errors; filter by `success` for downstream processing.

In [ ]:
import pandas as pd
from itertools import islice

sample_rows = []
for file in islice(worker_checkpoint_files(WORKER_ID), 3):
    sample_rows.extend(islice(read_drive_jsonl(file), 10))

sample_df = pd.DataFrame(sample_rows)
print(f"Preview: {len(sample_df)} records from at most three checkpoint files.")
if not sample_df.empty:
    display(sample_df.head(10))
else:
    print("No uploaded checkpoints yet.")
